# 数据预处理

## 导入库

In [3]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
import re
import json
import sqlalchemy.types as sqltypes

## Pandas 显示设置

In [5]:
# 设置显示选项，便于在控制台查看大表数据
pd.set_option("display.max_columns", None)      # 显示所有列
pd.set_option("display.width", 200)            # 控制台显示宽度
pd.set_option("display.max_colwidth", 200)     # 每列最大显示长度

## 数据库配置

In [7]:
# 用于从 MySQL 读取原始数据并写入处理后的数据表
DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "123456",
    "database": "shixiseng_db",
    "charset": "utf8"  # 数据库字符集，utf8_general_ci 排序规则
}

SOURCE_TABLE = "shixiseng_job_details"
PROCESSED_TABLE = "processed_job_details"

# 创建 SQLAlchemy 引擎，用于数据读写
engine = create_engine(
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset=utf8"
)

## 读取原始数据

In [9]:
# 将原始招聘数据表加载到 DataFrame，便于后续分析和处理
df = pd.read_sql(f"SELECT * FROM {SOURCE_TABLE}", engine)

print(f"原始数据行数: {len(df)}, 列数: {df.shape[1]}")
print(df.head())

原始数据行数: 172063, 列数: 30
   id group_name item_text                                                                                                                      item_url  search_page         intern_id  \
0   1         产品      产品助理  https://www.shixiseng.com/interns?keyword=%E4%BA%A7%E5%93%81%E5%8A%A9%E7%90%86&city=%E5%85%A8%E5%9B%BD&type=intern&from=menu            1  inn_83amnc5w13bs   
1   2         产品        产品                    https://www.shixiseng.com/interns?keyword=%E4%BA%A7%E5%93%81&city=%E5%85%A8%E5%9B%BD&type=intern&from=menu            1  inn_hpywzaipk8ss   
2   3         产品      产品经理  https://www.shixiseng.com/interns?keyword=%E4%BA%A7%E5%93%81%E7%BB%8F%E7%90%86&city=%E5%85%A8%E5%9B%BD&type=intern&from=menu            1  inn_hpywzaipk8ss   
3   4         产品      产品设计  https://www.shixiseng.com/interns?keyword=%E4%BA%A7%E5%93%81%E8%AE%BE%E8%AE%A1&city=%E5%85%A8%E5%9B%BD&type=intern&from=menu            1  inn_g0jzbk3qp6y8   
4   5         产品      产品助理  https://www.sh

## 删除无关列

In [11]:
# 删除前的列名
print("删除前的列名：")
print(df.columns.tolist())

# 去掉分析中不需要的列，减少冗余数据
# 包括 URL、ID、时间戳、文本描述等，避免存储和计算冗余
columns_to_drop = [
    'item_url', 'id', 'search_page', 'intern_id', 'detail_url',
    'resume_language', 'job_address', 'job_description',
    'company_desc_detail', 'company_tags_detail',
    'company_intro_img_links', 'created_at', 'updated_at'
]
df.drop(columns=columns_to_drop, inplace=True)

# 删除后的列名
print("删除后的列名：")
print(df.columns.tolist())

删除前的列名：
['id', 'group_name', 'item_text', 'item_url', 'search_page', 'intern_id', 'detail_url', 'job_title_detail', 'publish_time_detail', 'salary_detail', 'city_detail', 'degree_detail', 'days_per_week_detail', 'intern_months_detail', 'job_tags_detail', 'job_header_img_links', 'job_description', 'resume_language', 'deadline_detail', 'job_address', 'company_name_detail', 'company_desc_detail', 'company_tags_detail', 'industry_detail', 'company_nature_detail', 'company_size_detail', 'company_location_detail', 'company_intro_img_links', 'created_at', 'updated_at']
删除后的列名：
['group_name', 'item_text', 'job_title_detail', 'publish_time_detail', 'salary_detail', 'city_detail', 'degree_detail', 'days_per_week_detail', 'intern_months_detail', 'job_tags_detail', 'job_header_img_links', 'deadline_detail', 'company_name_detail', 'industry_detail', 'company_nature_detail', 'company_size_detail', 'company_location_detail']


## 生成“急招”标志列

In [13]:
# 先生成急招标志列
df['is_urgent_hire'] = df['job_header_img_links'].fillna('').str.contains(
    "https://sxsimg.xiaoyuanzhao.com/static/new_main/img/ji.svg"
).astype(int)

print("急招标志生成结果预览：")
print(df[['job_header_img_links', 'is_urgent_hire']].head(10))

急招标志生成结果预览：
                                         job_header_img_links  is_urgent_hire
0                                                                           0
1  https://sxsimg.xiaoyuanzhao.com/static/new_main/img/ji.svg               1
2  https://sxsimg.xiaoyuanzhao.com/static/new_main/img/ji.svg               1
3                                                                           0
4                                                                           0
5  https://sxsimg.xiaoyuanzhao.com/static/new_main/img/ji.svg               1
6  https://sxsimg.xiaoyuanzhao.com/static/new_main/img/ji.svg               1
7                                                                           0
8                                                                           0
9                                                                           0


In [14]:
# 确认无误后再删除原列
df.drop(columns='job_header_img_links', inplace=True)

## 岗位标签列

In [16]:
print("job_tags_detail 原始内容前5行：")
print(df['job_tags_detail'].head())

job_tags_detail 原始内容前5行：
0    可转正实习、接受大一大二、零基础实习、提供免费住宿、一对一导师、免费班车、周末双休
1                                         周末双休
2                                         周末双休
3                             学习机会多、成长空间大、免费三餐
4              地铁周边、周末双休、一对一导师、留学生实习、节日福利、产品开发
Name: job_tags_detail, dtype: object


In [17]:
# 拆分成列表
df['job_tags'] = df['job_tags_detail'].fillna('').str.split('、')

In [18]:
print("拆分为列表后的 job_tags 前5行：")
print(df['job_tags'].head())

拆分为列表后的 job_tags 前5行：
0    [可转正实习, 接受大一大二, 零基础实习, 提供免费住宿, 一对一导师, 免费班车, 周末双休]
1                                               [周末双休]
2                                               [周末双休]
3                                 [学习机会多, 成长空间大, 免费三餐]
4               [地铁周边, 周末双休, 一对一导师, 留学生实习, 节日福利, 产品开发]
Name: job_tags, dtype: object


In [19]:
df['job_tags'] = df['job_tags'].apply(lambda x: json.dumps(x, ensure_ascii=False))  # JSON 存储并保留中文

In [20]:
print("转为 JSON 后的 job_tags 前5行：")
print(df['job_tags'].head())

转为 JSON 后的 job_tags 前5行：
0    ["可转正实习", "接受大一大二", "零基础实习", "提供免费住宿", "一对一导师", "免费班车", "周末双休"]
1                                                           ["周末双休"]
2                                                           ["周末双休"]
3                                         ["学习机会多", "成长空间大", "免费三餐"]
4                 ["地铁周边", "周末双休", "一对一导师", "留学生实习", "节日福利", "产品开发"]
Name: job_tags, dtype: object


In [21]:
# 删除原始列
df.drop(columns='job_tags_detail', inplace=True)

## 薪资列处理

In [23]:
print("salary_detail 原始内容前10行：")
df['salary_detail'].head(10)

salary_detail 原始内容前10行：


0    150-300/天
1         薪资面议
2         薪资面议
3    120-150/天
4    120-140/天
5         薪资面议
6         薪资面议
7    150-180/天
8        100/天
9    100-200/天
Name: salary_detail, dtype: object

In [24]:
salary_col = df['salary_detail'].fillna('')

# is_salary_negotiable: 是否薪资面议，1 表示面议，0 表示固定薪资
df['is_salary_negotiable'] = salary_col.isin(['薪资面议', '', 'nan', 'None']).astype(int)

# 向量化提取数字区间，提高处理效率
salary_clean = salary_col.str.replace('/天', '', regex=False)
range_match = salary_clean.str.extract(r'(\d+)-(\d+)').astype(float)
single_match = salary_clean.str.extract(r'^(\d+)$').astype(float)

# salary_min_per_day / salary_max_per_day: 日薪最小值和最大值
df['salary_min_per_day'] = range_match[0].combine_first(single_match[0])
df['salary_max_per_day'] = range_match[1].combine_first(single_match[0])

In [25]:
print("薪资字段处理结果前10行：")
df[['salary_detail', 'is_salary_negotiable', 'salary_min_per_day', 'salary_max_per_day']].head(10)

薪资字段处理结果前10行：


,salary_detail,is_salary_negotiable,salary_min_per_day,salary_max_per_day
0,150-300/天,0,150.0,300.0
1,薪资面议,1,NaN,NaN
2,薪资面议,1,NaN,NaN
3,120-150/天,0,120.0,150.0
4,120-140/天,0,120.0,140.0
5,薪资面议,1,NaN,NaN
6,薪资面议,1,NaN,NaN
7,150-180/天,0,150.0,180.0
8,100/天,0,100.0,100.0
9,100-200/天,0,100.0,200.0


In [26]:
df.drop(columns='salary_detail', inplace=True)

## 实习月份处理

In [28]:
print("intern_months_detail 原始内容前10行：")
df['intern_months_detail'].head(10)

intern_months_detail 原始内容前10行：


0    实习3个月
1    实习3个月
2    实习3个月
3    实习3个月
4    实习3个月
5    实习3个月
6    实习3个月
7    实习6个月
8    实习3个月
9    实习3个月
Name: intern_months_detail, dtype: object

In [29]:
# 提取实习时长（月数）
df['internship_months'] = df['intern_months_detail'].str.extract(r'(\d+)').astype(int)

In [30]:
print("实习时长提取结果前10行：")
df[['intern_months_detail', 'internship_months']].head(10)

实习时长提取结果前10行：


,intern_months_detail,internship_months
0,实习3个月,3
1,实习3个月,3
2,实习3个月,3
3,实习3个月,3
4,实习3个月,3
5,实习3个月,3
6,实习3个月,3
7,实习6个月,6
8,实习3个月,3
9,实习3个月,3


In [31]:
df.drop(columns='intern_months_detail', inplace=True)

## 每周工作天数列

In [33]:
print("days_per_week_detail 原始内容前10行：")
print(df['days_per_week_detail'].head(10))

days_per_week_detail 原始内容前10行：
0    4天／周
1    5天／周
2    5天／周
3    5天／周
4    4天／周
5    5天／周
6    5天／周
7    5天／周
8    5天／周
9    5天／周
Name: days_per_week_detail, dtype: object


In [34]:
# 提取每周工作天数
df['days_per_week'] = df['days_per_week_detail'].fillna('0').str.extract(r'(\d+)').astype(int)

In [35]:
print("每周工作天数提取结果前10行：")
print(df[['days_per_week_detail', 'days_per_week']].head(10))

每周工作天数提取结果前10行：
  days_per_week_detail  days_per_week
0                 4天／周              4
1                 5天／周              5
2                 5天／周              5
3                 5天／周              5
4                 4天／周              4
5                 5天／周              5
6                 5天／周              5
7                 5天／周              5
8                 5天／周              5
9                 5天／周              5


In [36]:
df.drop(columns='days_per_week_detail', inplace=True)

## 写入数据库

In [38]:
dtype_mapping = {
    'job_tags': sqltypes.Text(collation='utf8_general_ci'),
    'job_title_detail': sqltypes.VARCHAR(255, collation='utf8_general_ci'),
    'item_text': sqltypes.VARCHAR(255, collation='utf8_general_ci'),
    'company_name_detail': sqltypes.VARCHAR(255, collation='utf8_general_ci')
}

df.to_sql(
    PROCESSED_TABLE, 
    engine, 
    if_exists='replace',
    index=False, 
    dtype=dtype_mapping
)

print(f"处理后的数据已成功写入数据库表: {PROCESSED_TABLE}")

处理后的数据已成功写入数据库表: processed_job_details
